In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load raw datasets
events = pd.read_csv(
    "E:/Product_Analyst/Data/raw/product_app_events_raw.csv"
)

orders = pd.read_csv(
    "E:/Product_Analyst/Data/raw/product_orders_raw.csv"
)

print("Events:", events.shape)
print("Orders:", orders.shape)

Events: (12035, 12)
Orders: (3612, 16)


In [2]:
events_clean = events.drop_duplicates().copy()
orders_clean = orders.drop_duplicates().copy()

print("Events before:", len(events))
print("Events after:", len(events_clean))

print("Orders before:", len(orders))
print("Orders after:", len(orders_clean))

Events before: 12035
Events after: 12000
Orders before: 3612
Orders after: 3600


In [5]:
events_clean["event_timestamp"] = pd.to_datetime(
    events_clean["event_timestamp"],
    errors="coerce"
)

orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    errors="coerce"
)

print(events_clean["event_timestamp"].dtype)
print(orders_clean["order_date"].dtype)

datetime64[ns]
datetime64[ns]


In [6]:
events_clean["event_name"] = (
    events_clean["event_name"]
    .str.strip()
    .str.lower()
)

print(events_clean["event_name"].value_counts())

view_item           3650
app_open            2131
add_to_cart         1869
search              1386
begin_checkout      1098
remove_from_cart     954
purchase             912
Name: event_name, dtype: int64


In [7]:
print("EVENT CITIES")
display(events_clean["city"].value_counts(dropna=False))

print("ORDER CITIES")
display(orders_clean["city"].value_counts(dropna=False))

EVENT CITIES


AHD              635
Jaipur           604
Kolkata          598
JAIPUR           597
PUNE             597
Ahmedabad        593
Pune             562
Calcutta         545
Vizag            424
Bombay           419
Chennai          416
Hyderabad        415
Delhi            409
chennai          400
Visakhapatnam    399
MUMBAI           397
HYD              394
hyderabad        393
MADRAS           384
Mumbai           382
New Delhi        382
DELHI            376
VISAKHAPATNAM    362
Bengaluru        308
bengaluru        297
BANGALORE        281
Bangalore        276
NaN              155
Name: city, dtype: int64

ORDER CITIES


Calcutta         199
AHD              195
PUNE             191
Kolkata          190
JAIPUR           187
Jaipur           178
Pune             172
Ahmedabad        160
Mumbai           133
DELHI            133
Hyderabad        133
Bombay           124
MUMBAI           120
HYD              118
Visakhapatnam    118
Vizag            117
Chennai          116
chennai          115
New Delhi        115
Delhi            114
MADRAS           108
VISAKHAPATNAM    104
hyderabad        100
bengaluru         87
Bangalore         83
BANGALORE         80
Bengaluru         78
NaN               32
Name: city, dtype: int64

In [8]:
city_mapping = {
    "Bangalore": "Bengaluru",
    "BANGALORE": "Bengaluru",
    "bengaluru ": "Bengaluru",
    "HYD": "Hyderabad",
    "hyderabad": "Hyderabad",
    "MADRAS": "Chennai",
    "chennai ": "Chennai",
    "MUMBAI": "Mumbai",
    "Bombay": "Mumbai",
    "PUNE": "Pune",
    "New Delhi": "Delhi",
    "DELHI": "Delhi",
    "Calcutta": "Kolkata",
    "Visakhapatnam": "Vizag",
    "VISAKHAPATNAM": "Vizag",
    "JAIPUR": "Jaipur",
    "AHD": "Ahmedabad"
}

In [9]:
for df in [events_clean, orders_clean]:

    df["city"] = (
        df["city"]
        .str.strip()
        .replace(city_mapping)
    )

In [10]:
display(events_clean["city"].value_counts(dropna=False))
display(orders_clean["city"].value_counts(dropna=False))

Ahmedabad    1228
Hyderabad    1202
Jaipur       1201
Mumbai       1198
Vizag        1185
Delhi        1167
Pune         1159
Kolkata      1143
Bengaluru     865
Chennai       800
chennai       400
bengaluru     297
NaN           155
Name: city, dtype: int64

Kolkata      389
Mumbai       377
Jaipur       365
Pune         363
Delhi        362
Ahmedabad    355
Hyderabad    351
Vizag        339
Bengaluru    241
Chennai      224
chennai      115
bengaluru     87
NaN           32
Name: city, dtype: int64

In [11]:
city_mapping = {
    "bangalore": "Bengaluru",
    "bengaluru": "Bengaluru",
    "hyd": "Hyderabad",
    "hyderabad": "Hyderabad",
    "madras": "Chennai",
    "chennai": "Chennai",
    "mumbai": "Mumbai",
    "bombay": "Mumbai",
    "pune": "Pune",
    "new delhi": "Delhi",
    "delhi": "Delhi",
    "calcutta": "Kolkata",
    "kolkata": "Kolkata",
    "vizag": "Vizag",
    "visakhapatnam": "Vizag",
    "jaipur": "Jaipur",
    "ahd": "Ahmedabad",
    "ahmedabad": "Ahmedabad"
}

for df in [events_clean, orders_clean]:

    normalized_city = (
        df["city"]
        .str.strip()
        .str.lower()
    )

    df["city"] = normalized_city.map(city_mapping)

In [12]:
print("CLEANED EVENT CITIES")

display(events_clean["city"].value_counts(dropna=False))

print("\nCLEANED ORDER CITIES")

display(orders_clean["city"].value_counts(dropna=False))

CLEANED EVENT CITIES


Ahmedabad    1228
Hyderabad    1202
Jaipur       1201
Chennai      1200
Mumbai       1198
Vizag        1185
Delhi        1167
Bengaluru    1162
Pune         1159
Kolkata      1143
NaN           155
Name: city, dtype: int64


CLEANED ORDER CITIES


Kolkata      389
Mumbai       377
Jaipur       365
Pune         363
Delhi        362
Ahmedabad    355
Hyderabad    351
Chennai      339
Vizag        339
Bengaluru    328
NaN           32
Name: city, dtype: int64

In [13]:
print("EVENTS MISSING VALUES")

display(
    events_clean.isnull().sum()[
        events_clean.isnull().sum() > 0
    ]
)

print("\nORDERS MISSING VALUES")

display(
    orders_clean.isnull().sum()[
        orders_clean.isnull().sum() > 0
    ]
)


EVENTS MISSING VALUES


product_id      3571
product_name    3517
category        3517
city             155
device            91
dtype: int64


ORDERS MISSING VALUES


city            32
discount_pct    24
dtype: int64

In [14]:
missing_pid = events_clean[
    events_clean["product_id"].isnull()
]

print("Total missing product IDs:", len(missing_pid))

print(
    "Product name available:",
    missing_pid["product_name"].notnull().sum()
)

print(
    "Category available:",
    missing_pid["category"].notnull().sum()
)

print(
    "Both name and category available:",
    (
        missing_pid["product_name"].notnull() &
        missing_pid["category"].notnull()
    ).sum()
)

Total missing product IDs: 3571
Product name available: 54
Category available: 54
Both name and category available: 54


In [15]:
missing_device = events_clean[
    events_clean["device"].isnull()
]

display(
    missing_device[
        ["event_name", "user_id", "session_id",
         "city", "traffic_source"]
    ].head(15)
)

,event_name,user_id,session_id,city,traffic_source
64,remove_from_cart,U00593,S002264,Chennai,Social Media
95,app_open,U00904,S001731,Mumbai,Social Media
193,add_to_cart,U01660,S005031,Kolkata,Email
208,app_open,U00338,S001919,Kolkata,Direct
304,view_item,U02356,S002592,Kolkata,Email
331,search,U00798,S001389,Mumbai,Organic Search
628,add_to_cart,U02181,S003200,Jaipur,Social Media
1046,view_item,U01090,S002981,Pune,Email
1152,begin_checkout,U00929,S003022,Delhi,Organic Search
1179,begin_checkout,U00837,S001261,Pune,Organic Search


In [16]:
product_mapping = {
    "Wireless Earbuds": "P1001",
    "Smart Watch": "P1002",
    "Bluetooth Speaker": "P1003",
    "Running Shoes": "P2001",
    "Cotton T-Shirt": "P2002",
    "Backpack": "P2003",
    "Mixer Grinder": "P3001",
    "Water Bottle": "P3002",
    "Bedsheet Set": "P3003",
    "Face Serum": "P4001",
    "Sunscreen SPF 50": "P4002",
    "Yoga Mat": "P5001",
    "Cricket Bat": "P5002",
    "Data Analytics Handbook": "P6001",
    "Python Programming": "P6002"
}

In [17]:
product_check = (
    events_clean[
        events_clean["product_name"].notna()
    ][["product_name", "product_id"]]
    .drop_duplicates()
)

display(product_check)

,product_name,product_id
0,Bluetooth Speaker,P1003
1,Mixer Grinder,P3001
2,Backpack,P2003
3,Smart Watch,P1002
4,Sunscreen SPF 50,P4002
5,Cricket Bat,P5002
8,Bedsheet Set,P3003
10,Wireless Earbuds,P1001
19,Face Serum,P4001
21,Python Programming,P6002


In [18]:
missing_id_mask = (
    events_clean["product_id"].isna() &
    events_clean["product_name"].notna()
)

events_clean.loc[
    missing_id_mask, "product_id"
] = events_clean.loc[
    missing_id_mask, "product_name"
].map(product_mapping)

print(
    "Remaining missing product IDs:",
    events_clean["product_id"].isna().sum()
)

Remaining missing product IDs: 3517


In [19]:
events_clean["product_id_recovered"] = (
    events_clean["product_id"].notna() &
    events_clean["product_name"].map(product_mapping).notna() &
    events_clean["product_name"].notna()
)

print(
    "Recovered product IDs:",
    events_clean["product_id_recovered"].sum()
)

Recovered product IDs: 8483


In [20]:
product_missing_report = (
    events_clean.groupby("event_name")
    .agg(
        total_events=("event_id", "count"),
        missing_product_id=(
            "product_id",
            lambda x: x.isna().sum()
        )
    )
)

product_missing_report["missing_percentage"] = (
    product_missing_report["missing_product_id"]
    / product_missing_report["total_events"] * 100
).round(2)

display(product_missing_report)

,total_events,missing_product_id,missing_percentage
event_name,,,
add_to_cart,1869,0,0.0
app_open,2131,2131,100.0
begin_checkout,1098,0,0.0
purchase,912,0,0.0
remove_from_cart,954,0,0.0
search,1386,1386,100.0
view_item,3650,0,0.0


In [21]:
missing_id_mask = (
    events_clean["product_id"].isna()
    & events_clean["product_name"].notna()
)

events_clean["product_id_recovered"] = False

events_clean.loc[
    missing_id_mask, "product_id"
] = events_clean.loc[
    missing_id_mask, "product_name"
].map(product_mapping)

events_clean.loc[
    missing_id_mask & events_clean["product_id"].notna(),
    "product_id_recovered"
] = True

In [22]:
# Reconstruct original missing product IDs
original_events = events.drop_duplicates().copy()

original_missing_ids = set(
    original_events.loc[
        original_events["product_id"].isna()
        & original_events["product_name"].notna(),
        "event_id"
    ]
)

# Mark only genuinely recovered records
events_clean["product_id_recovered"] = (
    events_clean["event_id"].isin(original_missing_ids)
    & events_clean["product_id"].notna()
)

print(
    "Actually recovered:",
    events_clean["product_id_recovered"].sum()
)

print(
    "Still missing:",
    events_clean["product_id"].isna().sum()
)


Actually recovered: 54
Still missing: 3517


In [23]:
orders_clean["calculated_discount"] = (
    (
        orders_clean["gross_amount_inr"]
        - orders_clean["net_amount_inr"]
    )
    / orders_clean["gross_amount_inr"]
) * 100

# Create an audit flag
orders_clean["discount_imputed"] = (
    orders_clean["discount_pct"].isna()
)

# Preserve original discount values
orders_clean["discount_pct_clean"] = (
    orders_clean["discount_pct"]
)

# Recover missing discounts
mask = orders_clean["discount_pct_clean"].isna()

orders_clean.loc[
    mask, "discount_pct_clean"
] = orders_clean.loc[
    mask, "calculated_discount"
].round(2)

In [24]:
print("Original missing discounts:",
      orders_clean["discount_pct"].isna().sum())

print("Recovered discounts:",
      orders_clean["discount_imputed"].sum())

print("Remaining missing discounts:",
      orders_clean["discount_pct_clean"].isna().sum())

Original missing discounts: 24
Recovered discounts: 24
Remaining missing discounts: 0


In [25]:
orders_clean["expected_net_amount"] = (
    orders_clean["gross_amount_inr"]
    * (1 - orders_clean["discount_pct_clean"] / 100)
).round(2)

orders_clean["net_amount_difference"] = (
    orders_clean["net_amount_inr"]
    - orders_clean["expected_net_amount"]
).abs()

orders_clean[
    [
        "gross_amount_inr",
        "discount_pct_clean",
        "net_amount_inr",
        "expected_net_amount",
        "net_amount_difference"
    ]
].head(10)

,gross_amount_inr,discount_pct_clean,net_amount_inr,expected_net_amount,net_amount_difference
0,1207.79,5.0,1147.40,1147.40,0.0
1,899.94,0.0,899.94,899.94,0.0
2,3100.13,20.0,2480.10,2480.10,0.0
3,3769.05,0.0,3769.05,3769.05,0.0
4,3612.73,0.0,3612.73,3612.73,0.0
5,2613.67,0.0,2613.67,2613.67,0.0
6,1584.74,0.0,1584.74,1584.74,0.0
7,398.95,0.0,398.95,398.95,0.0
8,2628.06,15.0,2233.85,2233.85,0.0
9,482.43,20.0,385.94,385.94,0.0


In [26]:
print(
    "Orders with negative or zero gross amount:",
    (orders_clean["gross_amount_inr"] <= 0).sum()
)

print(
    "Orders with invalid discount percentages:",
    (
        (orders_clean["discount_pct_clean"] < 0)
        | (orders_clean["discount_pct_clean"] > 100)
    ).sum()
)

print(
    "Orders with net amount discrepancies above ₹0.05:",
    (orders_clean["net_amount_difference"] > 0.05).sum()
)

print(
    "Maximum net amount difference:",
    orders_clean["net_amount_difference"].max()
)

Orders with negative or zero gross amount: 0
Orders with invalid discount percentages: 0
Orders with net amount discrepancies above ₹0.05: 0
Maximum net amount difference: 0.010000000000218279


In [27]:
print("EVENTS — Remaining Missing Values")
print(events_clean.isnull().sum())

print("\nORDERS — Remaining Missing Values")
print(orders_clean.isnull().sum())

EVENTS — Remaining Missing Values
event_id                   0
event_timestamp            0
user_id                    0
session_id                 0
event_name                 0
product_id              3517
product_name            3517
category                3517
city                     155
device                    91
traffic_source             0
screen_name                0
product_id_recovered       0
dtype: int64

ORDERS — Remaining Missing Values
order_id                  0
order_date                0
user_id                   0
product_id                0
product_name              0
category                  0
city                     32
quantity                  0
unit_price_inr            0
discount_pct             24
gross_amount_inr          0
net_amount_inr            0
payment_method            0
order_status              0
acquisition_channel       0
device                    0
calculated_discount       0
discount_imputed          0
discount_pct_clean        0
expected_

In [28]:
print("Events shape:", events_clean.shape)
print("Orders shape:", orders_clean.shape)

print("\nDuplicate events:", events_clean.duplicated().sum())
print("Duplicate orders:", orders_clean.duplicated().sum())

Events shape: (12000, 13)
Orders shape: (3600, 21)

Duplicate events: 0
Duplicate orders: 0


In [29]:
# Remove temporary audit and validation columns
events_clean = events_clean.drop(
    columns=["product_id_recovered"],
    errors="ignore"
)

orders_clean = orders_clean.drop(
    columns=[
        "calculated_discount",
        "discount_imputed",
        "expected_net_amount",
        "net_amount_difference"
    ],
    errors="ignore"
)

print("Events shape:", events_clean.shape)
print("Orders shape:", orders_clean.shape)

Events shape: (12000, 12)
Orders shape: (3600, 17)


In [30]:
import os

processed_path = "E:/Product_Analyst/Data/processed"

os.makedirs(processed_path, exist_ok=True)

events_clean.to_csv(
    os.path.join(processed_path, "product_app_events_clean.csv"),
    index=False
)

orders_clean.to_csv(
    os.path.join(processed_path, "product_orders_clean.csv"),
    index=False
)

print("Cleaned datasets exported successfully!")

Cleaned datasets exported successfully!


In [31]:
events_check = pd.read_csv(
    "E:/Product_Analyst/Data/processed/product_app_events_clean.csv"
)

orders_check = pd.read_csv(
    "E:/Product_Analyst/Data/processed/product_orders_clean.csv"
)

print("Events:", events_check.shape)
print("Orders:", orders_check.shape)

Events: (12000, 12)
Orders: (3600, 17)
